# Custom pipelines and plugins

NeSyDep's five stages are swappable by name. This notebook builds a custom
pipeline, caches intermediate artifacts, and registers a new sampler.

In [1]:
import pandas as pd
from nesydep import Pipeline
from nesydep.core.config import (
    StratifiedSampleConfig, LightweightCorrelationConfig, FDMinerConfig,
)

df = pd.read_csv('../tests/data/german_credit_sample.csv')

## Compose a pipeline by hand

`cache_dir` writes the sample, correlated sets and sub-tables to disk —
handy for reproducing runs or mining the same sub-tables with different
parameters.

In [2]:
pipe = Pipeline(
    sampler='stratified',
    correlation='lightweight',
    partitioner='vertical',
    miner='pfminer',
    algorithm='my-pipeline',
    cache_dir='cache_run1',
    configs={
        'sampler': StratifiedSampleConfig(ratio=0.5, seed=7),
        'correlation': LightweightCorrelationConfig(method='pearson', threshold=0.2),
        'miner': FDMinerConfig(support=5, confidence=0.95),
    },
)
result = pipe.run(df)
result.stats

[1/4] sampling (300 rows) ...
[2/4] correlation extraction ...
[3/4] partitioning ...
[4/4] mining (10 sub-tables) ...


{'n_rows': 300,
 'n_columns': 21,
 'sample_seconds': 0.08305899999686517,
 'sample_rows': 292,
 'correlation_seconds': 0.0034967999963555485,
 'correlated_sets': 12,
 'partition_seconds': 0.0019614000048022717,
 'n_subtables': 10,
 'mine_seconds': 0.009465199982514605}

## Register a custom stage

Any class implementing the stage protocol can be registered and used by name.
Third-party packages can do the same via entry points (see docs/plugins.md).

In [3]:
from nesydep.core.registry import SAMPLERS

@SAMPLERS.decorator('head')
class HeadSampler:
    """Take the first n rows (deterministic, for debugging)."""
    def sample(self, frame, config=None):
        n = getattr(config, 'ratio', 0.5) if config else 0.5
        return frame.head(max(1, round(len(frame) * n))).reset_index(drop=True)

print('samplers:', SAMPLERS.names())

# pyref-fd is an exact but unoptimised textbook miner — use small tables
toy = pd.read_csv('../tests/data/toy_addresses.csv')
pipe2 = Pipeline(sampler='head', miner='pyref-fd', algorithm='debug-run')
r2 = pipe2.run(toy)
print(f'{len(r2)} FDs on the toy table')

samplers: ['head', 'random', 'representative', 'stratified']
23 FDs on the toy table


[1/4] sampling (12 rows) ...
[2/4] correlation extraction ...
[3/4] partitioning ...
[4/4] mining (1 sub-tables) ...


## Evaluate against ground truth

In [4]:
import nesydep as nd

tane = nd.discover(df.iloc[:, :16], algo='tane', support=5, confidence=1.0)
m = nd.evaluate(result.fds, tane.fds)   # pipeline FDs vs exact TANE FDs
print(f'precision={m.precision:.2f} recall={m.recall:.2f} f1={m.f1:.2f}')

[1/4] sampling (300 rows) ...
[2/4] correlation extraction ...
[3/4] partitioning ...
[4/4] mining (1 sub-tables) ...


precision=0.00 recall=0.00 f1=0.00
